In [ ]:
# Databricks notebook source
# tutorial_name: 06-Day8-Medallion-PostRun-Checks
# notebookName: 06-Day8-Medallion-PostRun-Checks
# COMMAND ----------

# Databricks notebook source
# COMMAND ----------

# DBTITLE 1,Paths (same as Days 5–8)
notepoint_rel = "hands-on/day-08/notebooks/06-Day8-Medallion-PostRun-Checks.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
SOURCE_JSON = BASE_PATH + "/json/2015-summary.json"
DAY8_ROOT = f"{BASE_PATH}day08-{STUDENT_ID}"
MED_ROOT = f"{DAY8_ROOT}/medallion"
BRONZE_PATH = f"{MED_ROOT}/bronze_flight_routes"
SILVER_PATH = f"{MED_ROOT}/silver_flight_routes"
GOLD_PATH = f"{MED_ROOT}/gold_by_destination"
METRICS_PATH = f"{DAY8_ROOT}/workflow_metrics/run_metrics"
# COMMAND ----------
print("DAY8_ROOT:", DAY8_ROOT)
print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("OK: P_BASIC")
except Exception as e:
    print("P_BASIC:", e)
try:
    spark.read.format("csv").option("header", True).load(SOURCE_CSV).limit(1).collect()
    print("OK: SOURCE_CSV")
except Exception as e:
    print("SOURCE_CSV:", e)
try:
    spark.read.format("json").load(SOURCE_JSON).limit(1).collect()
    print("OK: SOURCE_JSON (optional)")
except Exception as e:
    print("(optional) SOURCE_JSON:", type(e).__name__)
# COMMAND ----------


## Post-run checks (after job or after running 03–05)

Run this notebook **once** when bronze, silver, and gold paths should all exist. Use it to verify counts, skim **Delta history**, and compare **Bronze** row count to **`P_BASIC`** (discussion only — schemas differ).


In [ ]:
from pyspark.sql import functions as F

def _safe_count(path):
    try:
        return spark.read.format("delta").load(path).count()
    except Exception as e:
        return f"(missing: {e})"

print("Bronze count:", _safe_count(BRONZE_PATH))
print("Silver count:", _safe_count(SILVER_PATH))
print("Gold count:", _safe_count(GOLD_PATH))

for label, pth in [
    ("bronze", BRONZE_PATH),
    ("silver", SILVER_PATH),
    ("gold", GOLD_PATH),
]:
    try:
        print("--- HISTORY:", label, "---")
        spark.sql(f"DESCRIBE HISTORY delta.`{pth}`").select("version", "operation").show(5, truncate=False)
    except Exception as e:
        print(label, "skip:", type(e).__name__)

try:
    n_basic = spark.read.format("delta").load(P_BASIC).count()
    n_bronze = spark.read.format("delta").load(BRONZE_PATH).count()
    print("P_BASIC rows:", n_basic)
    print("Bronze rows:", n_bronze)
except Exception as e:
    print("Compare skip:", e)
# COMMAND ----------


### Optional cleanup

Uncomment only on **your** prefix:

```python
# dbutils.fs.rm(DAY8_ROOT, recurse=True)
```